In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

df = pd.read_csv("../data/raw/student-mat.csv", sep=";")

# Target: pass (1) if final grade >= 10, else fail (0)
df["pass"] = (df["G3"] >= 10).astype(int)

In [2]:
target = "pass"

# Set A: all features except the target and G3 (which defines it)
X_all = df.drop(columns=["G3", "pass"])

# Set B: also drop G1 and G2, so the model must rely on other factors
X_no_grades = df.drop(columns=["G1", "G2", "G3", "pass"])

y = df[target]
print("Set A shape:", X_all.shape)
print("Set B shape:", X_no_grades.shape)

Set A shape: (395, 32)
Set B shape: (395, 30)


In [3]:
# Split once using row indices, so both feature sets use the SAME students
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=y
)

def split(X):
    return X.loc[train_idx], X.loc[test_idx]

y_train, y_test = y.loc[train_idx], y.loc[test_idx]
print("Train:", len(train_idx), " Test:", len(test_idx))
print("Pass rate in train:", round(y_train.mean(), 3))
print("Pass rate in test: ", round(y_test.mean(), 3))

Train: 316  Test: 79
Pass rate in train: 0.671
Pass rate in test:  0.671


In [4]:
def make_preprocessor(X):
    cat_cols = X.select_dtypes(include="object").columns.tolist()
    num_cols = X.select_dtypes(exclude="object").columns.tolist()
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])

In [5]:
def evaluate(X, name):
    X_train, X_test = split(X)

    dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
    dummy_pred = dummy.predict(X_test)

    model = Pipeline([
        ("prep", make_preprocessor(X)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    print(f"===== {name} =====")
    print("Dummy accuracy:", round((dummy_pred == y_test).mean(), 3))
    print("\nLogistic regression:")
    print(classification_report(y_test, pred, target_names=["fail", "pass"]))
    print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    print()

evaluate(X_all, "Set A: with G1 and G2")
evaluate(X_no_grades, "Set B: without G1 and G2")

===== Set A: with G1 and G2 =====
Dummy accuracy: 0.671

Logistic regression:
              precision    recall  f1-score   support

        fail       0.73      0.92      0.81        26
        pass       0.96      0.83      0.89        53

    accuracy                           0.86        79
   macro avg       0.84      0.88      0.85        79
weighted avg       0.88      0.86      0.86        79

ROC-AUC: 0.951
Confusion matrix:
 [[24  2]
 [ 9 44]]

===== Set B: without G1 and G2 =====
Dummy accuracy: 0.671

Logistic regression:
              precision    recall  f1-score   support

        fail       0.40      0.46      0.43        26
        pass       0.71      0.66      0.69        53

    accuracy                           0.59        79
   macro avg       0.56      0.56      0.56        79
weighted avg       0.61      0.59      0.60        79

ROC-AUC: 0.595
Confusion matrix:
 [[12 14]
 [18 35]]



C:\Users\lanie\AppData\Local\Temp\ipykernel_19572\876664248.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns.tolist()
C:\Users\lanie\AppData\Local\Temp\ipykernel_19572\876664248.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are excluded by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `exclude` to remove them, or to `include` to keep them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide